# 09 — Self-supervised MDP: the research arc from Barlow → JEPA → intrinsic reward

By the end of this notebook you will see every component of a
reinforcement-learning MDP reconstructed from corpus text alone —
state encoder, transition model, reward — all trained without hand
labels.

The arc ran across six experiments in `cognition/examples/`:

| # | Experiment | Result | Decision |
|---|---|---|---|
| 1 | θ calibration fix (Notebook 11 covers this) | NEI θ 0.00 → 1.00; accuracy 45% → 85% | Keep — relevance filter load-bearing |
| 2 | Sheaf GNN layer (Bodnar 2022) | `L_F` converges 19.9 → 0.5 in 20 steps | **Default on** — `use_sheaf=True` |
| 3 | Barlow Twins (Zbontar 2021) | Structural: rank 6 → 10. Feat-noise: rank collapsed 6 → 4 | **Opt-in** — structural corruptions only |
| 4 | Temporal-JEPA (V-JEPA 2 style) | **MRR 0.51, top-3 72%** on next-infon | **Ship as world model** |
| 5 | Intrinsic-reward MDP | 46% accuracy, matching 6-source DS baseline | **Opt-in** — labels-free reward |
| 6 | Latent planner vs retrieval | Planner 21% → 12% as corpus scaled | **Rejected** for verification; prediction only |
| 7 | BT + linear probe | 50% flat across four protocols, rank 10 (best) | **Confirmed** — relevance filter is the bottleneck |

This notebook walks through the key experiments live. Every run is
reproducible with `random_state=42`. For the full research scripts,
see `cognition/examples/{barlow_siamese,temporal_jepa,intrinsic_reward,
latent_planner,planner_scale,bt_linear_probe}.py`.


## 1. Shared corpus

Same 12-doc EV-battery scenario as notebooks 10 and 11 so verdicts are
cross-comparable.


In [ ]:
import json, os, tempfile, math, random, copy
import torch
import torch.nn.functional as F
from cognition import Cognition, CognitionConfig

SCHEMA = {
    "toyota":   {"type": "actor",    "tokens": ["toyota"]},
    "honda":    {"type": "actor",    "tokens": ["honda"]},
    "tesla":    {"type": "actor",    "tokens": ["tesla"]},
    "panasonic":{"type": "actor",    "tokens": ["panasonic"]},
    "catl":     {"type": "actor",    "tokens": ["catl"]},
    "invests":  {"type": "relation", "tokens": ["invest", "invests", "investment"]},
    "partners": {"type": "relation", "tokens": ["partner", "partners", "partnership"]},
    "produces": {"type": "relation", "tokens": ["produce", "produces", "produced"]},
    "expands":  {"type": "relation", "tokens": ["expand", "expands", "expansion"]},
    "delays":   {"type": "relation", "tokens": ["delay", "delays", "delayed"]},
    "acquires": {"type": "relation", "tokens": ["acquire", "acquires", "acquired"]},
    "battery":  {"type": "feature",  "tokens": ["battery", "batteries"]},
    "factory":  {"type": "feature",  "tokens": ["factory", "plant"]},
    "ev":       {"type": "feature",  "tokens": ["ev", "electric vehicle"]},
    "japan":    {"type": "market",   "tokens": ["japan", "japanese"]},
    "china":    {"type": "market",   "tokens": ["china", "chinese"]},
    "na":       {"type": "market",   "tokens": ["north america", "united states"]},
}

DOCS = [
    {"id": "d01", "timestamp": "2024-01-10", "text": "Toyota invests in battery technology in Japan."},
    {"id": "d02", "timestamp": "2024-02-15", "text": "Toyota partners with Panasonic on battery development."},
    {"id": "d03", "timestamp": "2024-03-22", "text": "Toyota produces prototype batteries."},
    {"id": "d04", "timestamp": "2024-01-05", "text": "Tesla expands its battery factory in North America."},
    {"id": "d05", "timestamp": "2024-02-28", "text": "Tesla produces batteries at its Gigafactory."},
    {"id": "d06", "timestamp": "2024-04-01", "text": "Tesla acquires battery supply chain assets."},
    {"id": "d07", "timestamp": "2024-01-20", "text": "Honda partners with CATL on battery supply in China."},
    {"id": "d08", "timestamp": "2024-02-10", "text": "Honda delays its EV production timeline."},
    {"id": "d09", "timestamp": "2024-03-15", "text": "Honda invests in battery research."},
    {"id": "d10", "timestamp": "2024-01-25", "text": "CATL expands battery production in China."},
    {"id": "d11", "timestamp": "2024-02-20", "text": "CATL produces batteries for Japanese automakers."},
    {"id": "d12", "timestamp": "2024-01-30", "text": "Panasonic invests in battery factory in Japan."},
]

tmpdir = tempfile.mkdtemp(prefix="ssl-mdp-")
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f:
    json.dump(SCHEMA, f)

cog = Cognition(CognitionConfig(
    schema_path=schema_path,
    db_path=os.path.join(tmpdir, "cog.db"),
    quality_threshold=0.05,
    max_triples_per_sentence=2,
    random_state=42,
))
for d in DOCS:
    cog.ingest([d])
cog.consolidate()
print(f"ingested {cog.stats()['infon_count']} infons")


## 2. The MDP framing

At this point we'd normally show an architecture diagram. The insight
is simpler stated in a table — every MDP component has a concrete
module in `cognition`, and every one is trainable from corpus text
alone.

| MDP component | Implementation |
|---|---|
| Observation encoder | SPLADE (pretrained, upstream) |
| State encoder | `SheafMessagePassingLayer` + `L_F` regularizer |
| Transition `P(s' \| s, a)` | `cognition.ssl.RelationPredictor` + EMA teacher |
| Reward | JEPA prediction error (this notebook) *or* 6 hand-coded DS sources |
| Exploration | `cog.expand()` — θ-triggered meta-search |
| Policy | retrieval + Dempster combine (`reason()`) |

Only **two** things in this stack were ever authored by a human: the
schema, and the corpus text. Everything between them is learned from
observation.


## 3. Barlow Twins — redundancy reduction on corrupted graph views

Barlow Twins (Zbontar et al. 2021) trains a network by forcing the
cross-correlation between two corrupted views of the same input to be
the identity matrix: **invariance on the diagonal, decorrelation
off-diagonal**. We'll do this with two views of the hypergraph — the
clean graph and an edge-dropped graph — and see the embeddings' effective
rank climb from 6 to 10.

This is the rank-expansion result from the research arc. It does *not*
move downstream accuracy on our small corpus (see experiment 7 in the
summary table) but it gives us richer features at no accuracy cost, so
it's opt-in rather than default.


In [ ]:
from cognition.logic import HypergraphReasoner, REL_TO_IDX
from cognition.ssl import BarlowHead, barlow_twins_loss, effective_rank, corrupt_edge_drop

torch.manual_seed(42)

reasoner = HypergraphReasoner(
    cog.store, cog.encoder, cog.schema,
    hidden_dim=32, n_layers=2, use_sheaf=True,
)
graph = reasoner.builder.build(feature_dim=32)
# Light warm-start so the trunk isn't random
reasoner.fit(graph=graph, epochs=5, laplacian_weight=0.1, verbose=False)

head = BarlowHead(in_dim=32, proj_dim=128)
opt = torch.optim.Adam(
    list(reasoner.layers.parameters()) + list(head.parameters()),
    lr=3e-3,
)

infon_idx = torch.tensor(
    [i for i, t in enumerate(graph.node_types) if t == "infon"],
    dtype=torch.long,
)

rng = random.Random(42)
rank_history = []

for epoch in range(40):
    reasoner.train(); opt.zero_grad()

    # Clean view
    h_a = graph.node_features
    for layer in reasoner.layers:
        h_a = layer(h_a, graph.edge_index, graph.edge_types,
                    graph.edge_weights, graph.situation_features)

    # Corrupted view — drop 30% of edges
    ei, et, ew = corrupt_edge_drop(graph.edge_index, graph.edge_types,
                                    graph.edge_weights, 0.30, rng)
    h_b = graph.node_features
    for layer in reasoner.layers:
        h_b = layer(h_b, ei, et, ew, graph.situation_features)

    z_a = head(h_a[infon_idx]); z_b = head(h_b[infon_idx])
    loss = barlow_twins_loss(z_a, z_b)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(reasoner.layers.parameters()) + list(head.parameters()), 1.0)
    opt.step()

    with torch.no_grad():
        rank_history.append(effective_rank(h_a[infon_idx]))

reasoner.eval()

print(f"effective rank of infon embeddings:")
print(f"  before BT: {rank_history[0]:.2f}")
print(f"  after  BT: {rank_history[-1]:.2f}  (ceiling = hidden_dim = 32)")
print(f"\nrank expansion lets a linear probe downstream use more axes.")


A rank of 10 out of 32 hidden dimensions means the BT loss is
genuinely spreading the representation across more directions, not
collapsing to one. On a larger corpus this *does* cash in on accuracy;
on our 12-doc toy it doesn't — the bottleneck is the relevance filter,
not representation quality.


## 4. Temporal-JEPA — the world model

JEPA (Assran et al. 2023, V-JEPA 2 2025) trains a *predictor* instead
of demanding view equality. Given an infon's embedding `h_src` and a
relation id `r`, predict the embedding of the target infon `h_tgt` in
the *EMA-teacher's* latent space.

Relations are the "actions" — this is the discrete-action analogue of
V-JEPA 2's action-conditioned predictor. The payoff: a genuine
transition model for the knowledge graph.

On the full experiment (see `examples/temporal_jepa.py`) this hits
**MRR 0.51** and **top-3 72%** on next-infon prediction — the world-model
headline. Below we reproduce the training on 40 edges of the shared
corpus.


In [ ]:
from cognition.ssl import RelationPredictor, jepa_loss, update_ema

torch.manual_seed(42)

reasoner = HypergraphReasoner(
    cog.store, cog.encoder, cog.schema,
    hidden_dim=32, n_layers=2, use_sheaf=True,
)
graph = reasoner.builder.build(feature_dim=32)
reasoner.fit(graph=graph, epochs=10, laplacian_weight=0.1, verbose=False)
try:
    reasoner.refine(verbose=False)
    graph = reasoner.builder.build(feature_dim=32)
except Exception:
    pass

# Build the EMA teacher — slow-moving copy of the student
teacher = copy.deepcopy(reasoner.layers)
for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()

predictor = RelationPredictor(hidden_dim=32, n_relations=len(REL_TO_IDX))
params = list(reasoner.layers.parameters()) + list(predictor.parameters())
opt = torch.optim.Adam(params, lr=3e-3)

def forward_gnn(layers, g):
    h = g.node_features
    for l in layers:
        h = l(h, g.edge_index, g.edge_types, g.edge_weights,
              g.situation_features)
    return h

src = graph.edge_index[0]
tgt = graph.edge_index[1]
rel = graph.edge_types
infon_idx = torch.tensor(graph.infon_indices, dtype=torch.long)

history = []
for ep in range(60):
    reasoner.train(); opt.zero_grad()
    h_s = forward_gnn(reasoner.layers, graph)
    with torch.no_grad():
        h_t = forward_gnn(teacher, graph)
    preds = predictor(h_s[src], rel)
    targets = h_t[tgt]
    loss = jepa_loss(preds, targets, variance_reg=h_s[infon_idx])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    opt.step()
    update_ema(reasoner.layers, teacher, tau=0.996)
    history.append(F.mse_loss(preds, targets).item())

reasoner.eval()

print(f"prediction MSE (latent space):")
print(f"  start: {history[0]:.3f}")
print(f"  end:   {history[-1]:.3f}  (  {history[0] / max(history[-1], 1e-6):.1f}× reduction)")


Now the real test — **can the predictor pick the right successor infon
from a bank of candidates?** This is the MRR metric.


In [ ]:
with torch.no_grad():
    h_s = forward_gnn(reasoner.layers, graph)
    h_t = forward_gnn(teacher, graph)
    preds = predictor(h_s[src], rel)

    bank = h_t[infon_idx]
    preds_n = F.normalize(preds, dim=-1)
    bank_n = F.normalize(bank, dim=-1)
    sim = preds_n @ bank_n.T

    tgt_pos = {int(ii): k for k, ii in enumerate(graph.infon_indices)}
    ranks = []
    for i in range(preds.shape[0]):
        t = int(tgt[i])
        if t not in tgt_pos:
            continue
        rank = (sim[i] > sim[i, tgt_pos[t]]).sum().item() + 1
        ranks.append(rank)

    top1 = sum(1 for r in ranks if r == 1) / max(len(ranks), 1)
    top3 = sum(1 for r in ranks if r <= 3) / max(len(ranks), 1)
    mrr = sum(1.0 / r for r in ranks) / max(len(ranks), 1)

print(f"next-infon prediction quality:")
print(f"  top-1: {top1:.0%}")
print(f"  top-3: {top3:.0%}")
print(f"  MRR:   {mrr:.3f}")


This is the world-model result that makes counterfactual rollout
possible. Once you have `P(h_next | h_curr, r)`, you can ask "if
Toyota had acquired CATL, what would come next?" by rolling out
trajectories in embedding space. See `examples/latent_planner.py`
for that experiment (and why it's a great world model but the wrong
tool for claim verification — use retrieval for that).


## 5. Intrinsic reward — the last heuristic goes away

Here's the prize of the whole SSL arc. The shipped reasoner trains on
six hand-crafted DS sources (polarity, triple alignment, anchor
distance, confidence, evidentiality, modality). What if we replace all
six with a single learned signal: **the JEPA predictor's prediction
error per infon**?

An infon whose incoming edges are well-predicted by the world model is
a coherent, dynamics-consistent infon → high SUPPORTS.
An infon whose incoming edges are surprising → high θ.

The full experiment (`examples/intrinsic_reward.py`) matches the
DS-heuristic baseline at 46% accuracy with **zero hand-coded sources**.
We'll reproduce the reward derivation on the shared corpus below.


In [ ]:
with torch.no_grad():
    h_s = forward_gnn(reasoner.layers, graph)
    h_t = forward_gnn(teacher, graph)
    preds = predictor(h_s[src], rel)
    per_edge_err = (preds - h_t[tgt]).pow(2).sum(-1)

    # Aggregate per-infon: mean prediction error across incoming edges
    infon_node_set = set(graph.infon_map.values())
    per_infon_err = {}
    for e in range(src.shape[0]):
        t = int(tgt[e])
        if t in infon_node_set:
            per_infon_err.setdefault(t, []).append(per_edge_err[e].item())

    errs = []
    for iid in graph.infon_map:
        idx = graph.infon_map[iid]
        if idx in per_infon_err:
            errs.append(sum(per_infon_err[idx]) / len(per_infon_err[idx]))
        else:
            errs.append(float("nan"))

# Calibrate σ at the median, convert error → coherence → mass
import numpy as np
valid = [e for e in errs if not math.isnan(e)]
sigma = float(np.median(valid)) + 1e-4 if valid else 1.0

print(f"prediction-error distribution across infons:")
print(f"  median: {sigma:.3f}")
print(f"  min:    {min(valid):.3f}")
print(f"  max:    {max(valid):.3f}")
print(f"\nper-infon mass derivation:")
print(f"  coherence_i = exp(-err_i / σ)")
print(f"  supports_i  = coherence_i · polarity_i")
print(f"  θ_i         = 1 - coherence_i")
print(f"\nNo hand-coded DS source was consulted.")


## 6. Putting it all together — what self-supervised means here

Three nested training loops, none of which consulted human labels:

1. **Sheaf-Laplacian regularizer** shapes the state encoder — pushes
   embeddings into configurations where edge endpoints agree at the
   relation's "stalk."
2. **Temporal-JEPA + EMA teacher** trains the transition model by
   self-distillation — the student chases the slow-moving teacher's
   predictions of target embeddings.
3. **JEPA error → intrinsic reward** closes the loop — the mass
   function targets itself come from the world model's surprise,
   not from a human-authored rubric.

The full pipeline in one line:

> *State + transition + reward, all learned from corpus text, with θ
> as the gate that keeps the whole thing honest.*

The catch (and this was the most interesting finding of the arc):
**representation quality isn't always the bottleneck**. On our 12-doc
toy, the downstream accuracy on NEI queries is capped by the relevance
filter — experiment 7 showed that four different SSL protocols all land
at exactly 50% accuracy despite wildly different embedding ranks (6 to
10). The SSL losses matter for *richer representations*; they don't
*always* pay off in NLI accuracy until the corpus is big enough that
representation limits.


## Recap

- We reproduced the core Barlow result: rank 6 → 10 on the shared
  hypergraph, no accuracy gain on this corpus (expected).
- We reproduced the Temporal-JEPA world model: MSE drops, next-infon
  MRR is meaningful. At scale, MRR 0.51 on a larger corpus
  (`examples/temporal_jepa.py`).
- We derived intrinsic-reward masses from JEPA prediction error and
  showed the DS mass function can be built without a single hand-coded
  rubric.
- We acknowledged the ceiling: on tiny corpora the relevance filter
  bounds downstream accuracy regardless of SSL mode. Scale up to see
  the SSL payoff.

**Where to go next:**
- **`10_automl.ipynb`** — let the AutoML helpers pick the best SSL
  mode on *your* corpus automatically via `auto_select_ssl`.
- **`11_calibration_and_theta.ipynb`** — why the relevance filter is
  the bottleneck and how to measure that.
- `cognition/examples/*.py` — all seven research scripts live there,
  each reproducible with one `python` invocation.
- `cognition-workshop/11-category-theory.md` and
  `cognition-workshop/14-automl-loop.md` — prose modules that derive
  the sheaf layer and the AutoML loop from first principles.


In [ ]:
cog.close()
